# 09. PCR Denoising

This notebook audits the copied Qwen-0.5B PCR summary table and shows how the public `src.pcr` utilities can be applied to the same archive.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

from src.pcr import estimate_sigma_from_cross_layer_variance, shrinkage_denoise_dataframe
from src.statistical_analysis import pcr_corrected_auc

pcr_table = pd.read_csv(DATA / 'analysis' / 'pcr_auc_comparison_qwen05b.csv')
unified = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
pcr_table
        


In [ ]:
layer0 = pcr_table[pcr_table['layer'] == 0]
print(layer0.to_string(index=False))

cot_l0 = unified[(unified['condition'] == 'cot') & (unified['layer'] == 0)].copy()
features = ['speed', 'dir_consistency', 'tortuosity', 'effective_dim', 'radius_of_gyration']
result = pcr_corrected_auc(cot_l0, features, denoiser='shrinkage')
print('Fresh shrinkage-PCR demo on copied layer-0 CoT subset:')
print(result)
        


In [ ]:
plt.figure(figsize=(8, 4))
plot_df = pcr_table[['layer', 'raw_auc', 'pcr_auc']].copy()
plot_df = plot_df.rename(columns={'pcr_auc': 'PCR AUC', 'raw_auc': 'Raw AUC'})
plot_df = plot_df.melt(id_vars='layer', var_name='series', value_name='auc')
sns.lineplot(data=plot_df, x='layer', y='auc', hue='series', marker='o')
plt.title('Qwen-0.5B PCR summary table')
plt.ylabel('ROC AUC')
plt.tight_layout()
plt.savefig(FIGURES / 'repro_09_pcr_denoising.png', dpi=300)
plt.show()
        
